In [1]:
import geopandas as gpd
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import subprocess
import os
from   shapely.geometry import Point, Polygon, MultiPolygon, box
from   PIL import Image
import glob
import warnings; warnings.filterwarnings("ignore")
plt.rcParams["font.family"] = "Times New Roman"

ERROR:tornado.general:Uncaught exception in ZMQStream callback
Traceback (most recent call last):
  File "/Users/shg096/Desktop/RiverLakeNetwork/env/RiverLakeEnv/lib/python3.9/site-packages/traitlets/traitlets.py", line 651, in get
    value = obj._trait_values[self.name]
KeyError: '_control_lock'

During handling of the above exception, another exception occurred:

Traceback (most recent call last):
  File "/Users/shg096/Desktop/RiverLakeNetwork/env/RiverLakeEnv/lib/python3.9/site-packages/zmq/eventloop/zmqstream.py", line 565, in _log_error
    f.result()
  File "/Users/shg096/Desktop/RiverLakeNetwork/env/RiverLakeEnv/lib/python3.9/site-packages/ipykernel/kernelbase.py", line 301, in dispatch_control
    async with self._control_lock:
  File "/Users/shg096/Desktop/RiverLakeNetwork/env/RiverLakeEnv/lib/python3.9/site-packages/traitlets/traitlets.py", line 706, in __get__
    return self.get(obj, cls)  # type:ignore[return-value]
  File "/Users/shg096/Desktop/RiverLakeNetwork/env/River

In [2]:
hydrolakes = gpd.read_file('/Volumes/F:/hydrography/hydrolakes/HydroLAKES_polys_v10_shp/HydroLAKES_polys_v10_shp/HydroLAKES_polys_v10.shp')

base_dir = "/Users/shg096/Desktop/LakeRiverOut/MERITBasins"

sections = {
    "a": f"{base_dir}/pfaf71a_corrected",
    "b": f"{base_dir}/pfaf71b_corrected",
    "c": f"{base_dir}/pfaf71c_corrected",
    "d": f"{base_dir}/pfaf71d_corrected",
}

data = {}

for k, folder in sections.items():
    data[k] = {
        "riv": gpd.read_file(f"{folder}/riv.gpkg"),
        "cat": gpd.read_file(f"{folder}/cat.gpkg"),
        "lake": gpd.read_file(f"{folder}/lake.gpkg"),
        "lake_org": hydrolakes.copy()
    }

In [3]:
xmin, xmax = -98.00, -94.00
ymin, ymax = 53.00, 55.00

margin = 2.0
xmin2, xmax2 = xmin - margin, xmax + margin
ymin2, ymax2 = ymin - margin, ymax + margin

buffer_size = 0.001

def clip_box(df):
    return df.cx[xmin2:xmax2, ymin2:ymax2]

In [4]:
for k in data.keys():

    data[k]["riv_clip"] = clip_box(data[k]["riv"])
    data[k]["cat_clip"] = clip_box(data[k]["cat"])
    data[k]["lake_clip"] = clip_box(data[k]["lake"])

    # buffer lakes slightly
    data[k]["lake_clip"]["geometry"] = data[k]["lake_clip"].geometry.buffer(buffer_size)

    bbox = gpd.GeoDataFrame(
        geometry=[box(xmin, ymin, xmax, ymax)],
        crs=data[k]["riv"].crs
    )

    print(f"\n=== Dataset {k} ===")
    print("Riv:", len(gpd.sjoin(data[k]["riv_clip"], bbox, how="inner", predicate="intersects")))
    print("Cat:", len(gpd.sjoin(data[k]["cat_clip"], bbox, how="inner", predicate="intersects")))
    print("Lake:", len(gpd.sjoin(data[k]["lake_clip"], bbox, how="inner", predicate="intersects")))


=== Dataset a ===
Riv: 838
Cat: 1069
Lake: 60

=== Dataset b ===
Riv: 998
Cat: 1230
Lake: 221

=== Dataset c ===
Riv: 836
Cat: 1132
Lake: 124

=== Dataset d ===
Riv: 936
Cat: 1233
Lake: 225


In [18]:
import matplotlib.pyplot as plt

letters = ["a", "b", "c", "d"]

for i, k in enumerate(["a", "b", "c", "d"]):

    fig, ax = plt.subplots(figsize=(8, 10))

    riv = data[k]["riv_clip"]
    cat = data[k]["cat_clip"]
    lake = data[k]["lake_clip"]

    cat.plot(
        ax=ax,
        facecolor="#F2F2F2",
        edgecolor="grey",
        linewidth=0.1,
        zorder=1
    )

    riv.plot(ax=ax, color="blue", linewidth=0.4, zorder=2)

    ax.set_xlim(xmin, xmax)
    ax.set_ylim(ymin, ymax)
    ax.set_aspect('equal')

    # -------------------------
    # Axis control per panel
    # -------------------------
    if letters[i] == "a":
        ax.set_xticklabels([" "] * len(ax.get_xticks()))   # empty x labels
        # keep y labels
    
    elif letters[i] == "b":
        ax.set_xticklabels([" "] * len(ax.get_xticks()))
        ax.set_yticklabels([" "] * len(ax.get_yticks()))
    
    elif letters[i] == "c":
        # keep both
        pass
    
    elif letters[i] == "d":
        ax.set_yticklabels([" "] * len(ax.get_yticks()))   # empty y labels

    # -------------------------
    # Panel label
    # -------------------------
    ax.text(
        xmin + 0.93*(xmax-xmin),
        ymax - 0.05*(ymax-ymin),
        f"({letters[i]})",
        fontsize=20,
        fontweight='bold'
    )

    filename = f"figure_{letters[i]}.png"
    plt.savefig(filename, dpi=300, bbox_inches="tight")
    plt.close()

    print(f"Saved {filename}")

Saved figure_a.png
Saved figure_b.png
Saved figure_c.png
Saved figure_d.png


In [19]:
from PIL import Image
import os

png_files = ["figure_a.png", "figure_b.png", "figure_c.png", "figure_d.png"]
images = [Image.open(png) for png in png_files]

# Assume all images same size
img_width, img_height = images[0].size

# Create blank canvas (2x2 grid)
merged_image = Image.new(
    "RGB",
    (img_width * 2, img_height * 2),
    (255, 255, 255)
)

# Paste images
positions = [
    (0, 0),                    # a
    (img_width, 0),            # b
    (0, img_height),           # c
    (img_width, img_height)    # d
]

for img, pos in zip(images, positions):
    merged_image.paste(img, pos)

# Save final
merged_image.save("Figure.png", dpi=(1200, 1200))

print("Saved Figure.png")

# Remove the png files
for file in png_files:
    if os.path.exists(file):
        os.remove(file)
        print(f"Removed {file}")
    else:
        print(f"{file} does not exist")

Saved Figure.png
Removed figure_a.png
Removed figure_b.png
Removed figure_c.png
Removed figure_d.png
